# ProX 第一阶段：逐步试跑文档级 keep/drop

这个 Notebook 刻意不封装成完整程序。每个步骤都可以单独执行、查看中间变量并修改。

建议顺序：先跑一条手写文本，确认模型原始输出；再读取少量 JSONL；最后才决定自己的批处理和保存逻辑。

## 1. 修改本地配置

公司离线环境把 `MODEL_PATH` 改成本地模型目录，并保持 `LOCAL_FILES_ONLY=True`。

In [ ]:
from pathlib import Path

MODEL_PATH = "models/web-doc-refining-lm"
LOCAL_FILES_ONLY = True

INPUT_PATH = Path("example_input.jsonl")
OUTPUT_PATH = Path("output/notebook_output.jsonl")
TEXT_KEY = "content"

MAX_NEW_TOKENS = 32
MODEL_CONTEXT_LENGTH = 2048

## 2. 检查 PyTorch 和 GPU

如果 `cuda_available` 是 `False`，先解决公司环境中的 PyTorch/CUDA 安装，不要继续加载模型。

In [ ]:
import torch

print("torch_version:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16_supported:", torch.cuda.is_bf16_supported())

## 3. 加载 tokenizer

这里先只加载 tokenizer。出错时可以单独判断模型目录是否完整、SentencePiece 是否安装。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=False,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
tokenizer.truncation_side = "right"

print("vocab_size:", tokenizer.vocab_size)
print("bos/eos/pad:", tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id)

## 4. 加载模型

32GB 单卡建议使用 BF16；不支持 BF16 时改用 FP16。模型约 0.35B 参数。

In [ ]:
from transformers import AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda" and torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
elif device.type == "cuda":
    model_dtype = torch.float16
else:
    model_dtype = torch.float32

print("device:", device)
print("dtype:", model_dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=model_dtype,
    low_cpu_mem_usage=True,
    local_files_only=LOCAL_FILES_ONLY,
).to(device)
model.eval()

## 5. 准备一条手写文本

先不要读公司数据。手写一条容易观察的文档，确认 prompt、token 和生成流程都正常。

In [ ]:
text = """Example News

Scientists announced a new result today.
The experiment was conducted over three years.

Copyright 2026 Example News"""

print(text)

## 6. 手动构造 Llama-2 prompt

模型仓库的 tokenizer 没有可靠的 chat template，因此这里直接展示并构造完整 prompt。

In [ ]:
SYSTEM_PROMPT = "You are a helpful, respectful and honest assistant."

prompt = (
    "<s>[INST] <<SYS>>\n"
    f"{SYSTEM_PROMPT}\n"
    "<</SYS>>\n\n"
    f"{text} [/INST]"
)

print(prompt)

## 7. Tokenize，并观察是否截断

In [ ]:
max_prompt_tokens = MODEL_CONTEXT_LENGTH - MAX_NEW_TOKENS
raw_prompt_token_count = len(
    tokenizer.encode(prompt, add_special_tokens=False)
)
was_truncated = raw_prompt_token_count > max_prompt_tokens

inputs = tokenizer(
    prompt,
    add_special_tokens=False,
    truncation=True,
    max_length=max_prompt_tokens,
    return_tensors="pt",
).to(device)

print("raw_prompt_token_count:", raw_prompt_token_count)
print("actual_input_tokens:", inputs["input_ids"].shape[1])
print("was_truncated:", was_truncated)

## 8. 生成一条原始程序

先只看模型真实输出，不急着写解析逻辑。

In [ ]:
with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

input_width = inputs["input_ids"].shape[1]
new_token_ids = generated[0, input_width:]
program = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

print("raw program:", repr(program))

## 9. 自己决定解析规则

下面是保守示例：只有标准 `keep/drop` 才接受，其他输出标记为 `unknown`，但保留原文。

In [ ]:
import re

normalized_program = program.strip().lower()
normalized_program = re.sub(r"[.!。！]+$", "", normalized_program).strip()

if normalized_program == "keep":
    decision = "keep"
elif normalized_program == "drop":
    decision = "drop"
else:
    decision = "unknown"

output_text = "" if decision == "drop" else text

print("decision:", decision)
print("output_text:", repr(output_text))

## 10. 读取少量 JSONL

先限制 `MAX_RECORDS`，避免第一次试跑直接读取全量。JSONL 字符串中的 `\n` 经 `json.loads` 后会恢复成真实换行。

In [ ]:
import gzip
import json

MAX_RECORDS = 20
records = []

open_input = gzip.open if INPUT_PATH.name.endswith(".gz") else open

with open_input(INPUT_PATH, "rt", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        if TEXT_KEY not in row:
            raise KeyError(f"第 {line_number} 行缺少字段 {TEXT_KEY!r}")
        if not isinstance(row[TEXT_KEY], str):
            raise TypeError(f"第 {line_number} 行的文本字段不是字符串")
        records.append(row)
        if len(records) >= MAX_RECORDS:
            break

print("loaded records:", len(records))
print("first record keys:", records[0].keys())
print("first text:")
print(records[0][TEXT_KEY])

## 11. 逐条试跑 JSONL

这里故意逐条生成，便于随时停止并检查。确认正确后再考虑 batch、异常重试和分片。

In [ ]:
results = []

for index, row in enumerate(records):
    current_text = row[TEXT_KEY]
    current_text = current_text.replace("\r\n", "\n").replace("\r", "\n")
    current_text = current_text.replace("\x00", "").strip()

    current_prompt = (
        "<s>[INST] <<SYS>>\n"
        f"{SYSTEM_PROMPT}\n"
        "<</SYS>>\n\n"
        f"{current_text} [/INST]"
    )

    original_token_count = len(
        tokenizer.encode(current_prompt, add_special_tokens=False)
    )
    current_truncated = original_token_count > max_prompt_tokens

    current_inputs = tokenizer(
        current_prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=max_prompt_tokens,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        current_generated = model.generate(
            **current_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    current_input_width = current_inputs["input_ids"].shape[1]
    current_new_ids = current_generated[0, current_input_width:]
    current_program = tokenizer.decode(
        current_new_ids, skip_special_tokens=True
    ).strip()

    value = re.sub(
        r"[.!。！]+$", "", current_program.strip().lower()
    ).strip()
    current_decision = value if value in {"keep", "drop"} else "unknown"

    output_row = dict(row)
    output_row["prox_doc_program"] = current_program
    output_row["prox_doc_decision"] = current_decision
    output_row["prox_doc_text"] = (
        "" if current_decision == "drop" else current_text
    )
    output_row["prox_doc_truncated"] = current_truncated
    results.append(output_row)

    print(
        index,
        row.get("id"),
        repr(current_program),
        current_decision,
        "truncated=", current_truncated,
    )

## 12. 人工查看结果

先逐条比较原文、原始程序和决策。不要在第一轮直接删除 drop 数据。

In [ ]:
for row in results:
    print("=" * 80)
    print("id:", row.get("id"))
    print("program:", repr(row["prox_doc_program"]))
    print("decision:", row["prox_doc_decision"])
    print("truncated:", row["prox_doc_truncated"])
    print("text:")
    print(row[TEXT_KEY][:1000])

## 13. 确认后写出小样本结果

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    for row in results:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved:", OUTPUT_PATH)

## 下一步

完成这里后，先统计模型真实输出形式和误删情况。只有逻辑稳定后，才需要考虑批量 padding、OOM 重试、分片和断点恢复。目录中的 `run_doc_filter.py` 是可选的全量运行参考，不影响你在 Notebook 中自由改动处理逻辑。